### PySpark Otomoto Demo 

Źródło danych: https://www.kaggle.com/datasets/szymoncyperski/car-sales-offers-from-otomotopl-2023 


In [3]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

**Teoria:** Powyżej importujemy niezbędne biblioteki. `SparkSession` to główny punkt wejścia do funkcjonalności DataFrame i SQL w Sparku (od wersji 2.0). Moduł `functions` dostarcza wbudowane funkcje operujące na kolumnach, a `matplotlib.pyplot` posłuży nam do późniejszej wizualizacji danych.


In [ ]:
spark = SparkSession.builder \
    .appName("Otomoto Demo") \
    .getOrCreate()


**Teoria:** Tworzymy sesję Sparka. `builder` używa wzorca projektowego Builder do skonfigurowania sesji. `getOrCreate()` tworzy nową sesję lub pobiera istniejącą, co jest bezpieczne przy wielokrotnym uruchamianiu notatnika.


In [ ]:
df = spark.read.option("header", True) \
    .option("delimiter", ";") \
    .option("inferSchema", False) \
    .csv("otomoto_offers_eng_23-04-2023.csv")


**Teoria:** Wczytywanie danych. Spark używa leniwego ewaluowania (lazy evaluation) - dane nie są fizycznie wczytywane w tym momencie, tworzony jest tylko plan wykonania (DAG). Ustawiamy `header=True` ponieważ nasz plik CSV ma nagłówki, oraz określamy separator jako średnik `;`.


In [ ]:
df.show()

**Teoria:** `show()` to akcja (action), która uruchamia wykonanie obliczeń w Sparku. Dopiero teraz plik jest odczytywany, a wynik prezentowany na ekranie.


In [ ]:
df.filter(F.col("vehicle_brand") == "Volvo").show()

In [ ]:
df = df.withColumn("price_num",
                   F.regexp_replace(F.col("price"), r"[^\d]", "").cast("double"))

df = df.withColumn("mileage_km",
                   F.regexp_replace(F.col("mileage"), r"[^\d]", "").cast("integer"))

df = df.withColumn("production_year_int",
                   F.regexp_replace(F.col("production_year"), r"[^\d]", "").cast("integer"))

df = df.withColumn("engine_cc",
                   F.regexp_replace(F.col("engine_displacement"), r"[^\d]", "").cast("integer"))

df = df.withColumn("power_hp",
                   F.regexp_replace(F.col("power"), r"[^\d]", "").cast("integer"))

df = df.withColumn("fuel_clean",
                   F.lower(F.trim(F.col("fuel_type"))))

In [ ]:
df.select("vehicle_brand", "vehicle_model", "price_num", "mileage_km",
          "production_year_int", "engine_cc", "power_hp", "fuel_clean") \
  .show(10, truncate=False)

**Teoria:** `select()` to transformacja, która działa jak w SQL - pozwala wybrać podzbiór kolumn. Zmniejsza to ilość przetwarzanych danych w dalszych krokach.


In [ ]:
avg_brand = df.groupBy("vehicle_brand") \
              .agg(F.round(F.avg("price_num"), 2).alias("avg_price")) \
              .orderBy(F.col("avg_price").desc())

print("Średnia cena per marka")
avg_brand.show(20, truncate=False)

In [ ]:
fuel_count = df.groupBy("fuel_clean").count()
print("Liczba ogłoszeń wg rodzaju paliwa")
fuel_count.show()

In [ ]:
df.createOrReplaceTempView("cars")

In [ ]:
# SQL: zależność mocy i pojemności od ceny
spark.sql("""
    SELECT vehicle_brand,
           ROUND(AVG(power_hp), 1) AS avg_power,
           ROUND(AVG(engine_cc), 1) AS avg_cc,
           ROUND(AVG(price_num), 1) AS avg_price
    FROM cars
    GROUP BY vehicle_brand
    ORDER BY avg_power DESC
""").show()

In [ ]:
df.groupBy("production_year_int") \
  .count() \
  .orderBy(F.col("production_year_int").desc()) \
  .show()

In [ ]:
# Średnia cena i przebieg per marka i model
df.groupBy("vehicle_brand", "vehicle_model") \
  .agg(
      F.round(F.avg("price_num"), 2).alias("avg_price"),
      F.round(F.avg("mileage_km"), 2).alias("avg_mileage")
  ) \
  .orderBy(F.col("avg_price").desc()) \
  .show(20, truncate=False)

In [ ]:
# zależność ceny od przebiegu
price_mileage = df.select("price_num", "mileage_km") \
                  .where((F.col("price_num").isNotNull()) & (F.col("mileage_km").isNotNull()))

In [ ]:
pdf_scatter = price_mileage.sample(fraction=0.1, seed=42).toPandas()

plt.figure(figsize=(8,5))
plt.scatter(pdf_scatter["mileage_km"], pdf_scatter["price_num"], s=6)
plt.title("Cena vs Przebieg")
plt.xlabel("Przebieg [km]")
plt.ylabel("Cena")
plt.tight_layout()
plt.savefig("scatter_price_mileage.png")

print("Wizualizacja scatter zapisana jako scatter_price_mileage.png")

**Teoria:** `toPandas()` to akcja, która zbiera (collect) wszystkie dane na partycjach roboczych i przesyła je na węzeł główny (Driver), konwertując do struktury Pandas DataFrame. Uwaga: Można tego używać tylko na małych zbiorach (po limitowaniu np. top 10), w przeciwnym razie braknie pamięci RAM na Driverze!


---
# Zadanie samodzielne: Analiza Przestępczości w Chicago

Poniżej znajduje się miejsce na realizację zadania z analizy danych przy użyciu PySpark na zbiorze *Chicago Crimes* (około 50 000 ostatnich zdarzeń). Twoim celem jest przygotowanie, wyczyszczenie oraz zanalizowanie tych danych z wykorzystaniem zaawansowanych optymalizacji dostępnych w Sparku.

### Wymagania:
1. **Wczytanie i Czyszczenie Danych:** Wczytaj pobrany plik `chicago_crimes_sample.csv`. Usuń ewentualne duplikaty, wiersze z brakami danych (szczególnie w kluczowych kolumnach) i odfiltruj/napraw błędne daty.
2. **UDF i Pora Dnia:** Dodaj nową kolumnę z klasyfikacją pory dnia (np. noc, dzień, wieczór) utworzoną za pomocą User Defined Function (UDF) w oparciu o godzinę z kolumny `Date`.
3. **Optymalizacja i Partycjonowanie:** Zoptymalizuj przetwarzanie. Zastanów się, w których momentach użyć `cache()`. Przy dołączaniu mniejszych tabel słownikowych (jeśli byś je tworzył), wykorzystaj *broadcast join*. Ostatecznie zapisz przefiltrowane dane do formatu **Parquet** z podziałem na partycje według roku (`Year`).
4. **Analiza i Plany Zapytań:** Przeprowadź analizę statystyczną przestępstw (np. jakiego typu przestępstwa są najpopularniejsze w konkretnych lokacjach, o konkretnym czasie). Wykorzystaj funkcję `.explain()` aby udokumentować plan zapytania Sparka dla najcięższej agregacji.
5. *(Opcjonalnie)* **Uczenie Maszynowe (MLlib):** Spróbuj zbudować i wytrenować prosty model wieloklasowy, przewidujący rodzaj przestępstwa (`Primary Type`) na podstawie innych atrybutów, jak lokacja, godzina, arrest itp.

In [2]:
import os, sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21.0.11"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYTHONUTF8"] = "1"

existing = SparkSession.getActiveSession()
if existing:
    existing.stop()

spark = SparkSession.builder \
    .appName("Chicago Crimes") \
    .master("local[*]") \
    .config("spark.driver.host", "localhost") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print("Spark version:", spark.version)


# Krok 1: Wczytanie i czyszczenie danych
df_crimes = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv("chicago_crimes_sample.csv")

print(f"Wierszy po wczytaniu: {df_crimes.count()}")

df_crimes = df_crimes \
    .withColumn("date_parsed",
                F.to_timestamp(F.col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS")) \
    .withColumn("hour", F.hour(F.col("date_parsed"))) \
    .withColumn("year_int", F.col("year").cast("integer")) \
    .withColumn("arrest_int",
                F.when(F.lower(F.col("arrest")) == "true", 1).otherwise(0)) \
    .withColumn("domestic_int",
                F.when(F.lower(F.col("domestic")) == "true", 1).otherwise(0))

df_crimes = df_crimes.dropDuplicates(["id"])

key_cols = ["id", "primary_type", "date_parsed", "location_description", "year"]
df_clean = df_crimes.dropna(subset=key_cols)
df_clean = df_clean.filter(F.col("year_int") >= 2000)

print(f"Wierszy po czyszczeniu: {df_clean.count()}")
df_clean.select("id", "date_parsed", "hour", "primary_type",
                "location_description", "arrest_int", "domestic_int", "year_int").show(5)


df_clean = df_clean.withColumn(
    "time_of_day",
    F.when(F.col("hour").isNull(), "nieznana")
     .when((F.col("hour") >= 22) | (F.col("hour") <= 5), "noc")
     .when(F.col("hour") <= 11, "ranek")
     .when(F.col("hour") <= 17, "dzien")
     .otherwise("wieczor")
)

print("Rozklad por dnia:")
df_clean.groupBy("time_of_day").count().orderBy("time_of_day").show()


# Krok 3a: Cache – buforowanie po czyszczeniu
df_clean.cache()
df_clean.count()
print("df_clean zbuforowany w pamieci Sparka.")


# Krok 3b: Broadcast join – mala tabela slownikowa powagi
severity_data = [
    ("HOMICIDE",            "krytyczne"),
    ("ASSAULT",             "wysokie"),
    ("BATTERY",             "wysokie"),
    ("ROBBERY",             "wysokie"),
    ("SEX OFFENSE",         "wysokie"),
    ("KIDNAPPING",          "wysokie"),
    ("THEFT",               "srednie"),
    ("BURGLARY",            "srednie"),
    ("MOTOR VEHICLE THEFT", "srednie"),
    ("ARSON",               "srednie"),
    ("CRIMINAL DAMAGE",     "niskie"),
    ("NARCOTICS",           "niskie"),
    ("OTHER OFFENSE",       "niskie"),
    ("DECEPTIVE PRACTICE",  "niskie"),
    ("CRIMINAL TRESPASS",   "niskie"),
]
severity_df = spark.createDataFrame(severity_data, ["primary_type", "severity"])

df_enriched = df_clean.join(F.broadcast(severity_df), on="primary_type", how="left") \
    .fillna({"severity": "inne"})

print("Przykladowe wiersze po broadcast join:")
df_enriched.select("primary_type", "severity", "time_of_day", "location_description").show(10)


# Krok 3c: Zapis do Parquet z partycjonowaniem po roku
output_path = "crimes_parquet"

df_enriched.write \
    .partitionBy("year") \
    .mode("overwrite") \
    .parquet(output_path)

print(f"Zapisano do '{output_path}' z partycjonowaniem po 'year'.")
df_parquet = spark.read.parquet(output_path)
print(f"Wierszy wczytanych z Parquet: {df_parquet.count()}")

# Krok 4: Analiza statystyczna + plan zapytania
agg_query = df_enriched \
    .groupBy("primary_type", "location_description", "time_of_day") \
    .agg(
        F.count("*").alias("liczba"),
        F.round(F.avg("arrest_int"), 3).alias("wskaznik_aresztu")
    ) \
    .orderBy(F.col("liczba").desc())

print("Plan zapytania (.explain())")
agg_query.explain()
print("\nTop 20: przestepstw wg lokalizacji i pory dnia")
agg_query.show(20, truncate=False)

print("Liczba przestepstw per rok")
df_enriched.groupBy("year_int").count().orderBy("year_int").show()

print("Wskaznik aresztowan wg lokalizacji (min. 100 incydentow)")
df_enriched.groupBy("location_description") \
    .agg(F.count("*").alias("liczba"),
         F.round(F.avg("arrest_int"), 3).alias("wskaznik_aresztu")) \
    .filter(F.col("liczba") >= 100) \
    .orderBy(F.col("wskaznik_aresztu").desc()) \
    .show(15, truncate=False)

print("Powaga przestepstw wg pory dnia")
df_enriched.groupBy("time_of_day", "severity") \
    .count() \
    .orderBy("time_of_day", F.col("count").desc()) \
    .show(30)

Spark version: 4.1.2
Wierszy po wczytaniu: 149848
Wierszy po czyszczeniu: 49803
+--------+-------------------+----+---------------+--------------------+----------+------------+--------+
|      id|        date_parsed|hour|   primary_type|location_description|arrest_int|domestic_int|year_int|
+--------+-------------------+----+---------------+--------------------+----------+------------+--------+
|14110344|2026-02-14 06:10:00|   6|        ASSAULT|           APARTMENT|         0|           1|    2026|
|14110362|2026-02-14 07:13:00|   7|CRIMINAL DAMAGE|VEHICLE NON-COMME...|         0|           0|    2026|
|14110367|2026-02-14 06:30:00|   6|          THEFT|              STREET|         0|           0|    2026|
|14110369|2026-02-14 07:10:00|   7|        BATTERY|            SIDEWALK|         1|           0|    2026|
|14110381|2026-02-14 05:00:00|   5|        BATTERY|           RESIDENCE|         0|           1|    2026|
+--------+-------------------+----+---------------+--------------------+